In [1]:
#Import the petting zoo environment and evaluate
from Multi_Ag_Environment import CustomEnvironment
from pettingzoo.test import parallel_api_test
env = CustomEnvironment()
parallel_api_test(env, num_cycles=1_000_000)

[25, 10, 46]
self.agents: ['satellite1', 'satellite2', 'satellite3']
Returning observations for: ['satellite1', 'satellite2', 'satellite3']
Expected agents: ['satellite1', 'satellite2', 'satellite3']
[50, 7, 80]
self.agents: ['satellite1', 'satellite2', 'satellite3']
Returning observations for: ['satellite1', 'satellite2', 'satellite3']
Expected agents: ['satellite1', 'satellite2', 'satellite3']
1
self.agents: ['satellite1', 'satellite2', 'satellite3']
Returning observations for: ['satellite1', 'satellite2', 'satellite3']
Expected agents: ['satellite1', 'satellite2', 'satellite3']
2
self.agents: ['satellite1', 'satellite2', 'satellite3']
Returning observations for: ['satellite1', 'satellite2', 'satellite3']
Expected agents: ['satellite1', 'satellite2', 'satellite3']
3
self.agents: ['satellite1', 'satellite2', 'satellite3']
Returning observations for: ['satellite1', 'satellite2', 'satellite3']
Expected agents: ['satellite1', 'satellite2', 'satellite3']
4
self.agents: ['satellite1', 'sat

/home/ethan/.local/lib/python3.10/site-packages/pettingzoo/test/parallel_test.py:88: UserWarning: Live agent was not given observation
  warnings.warn(f"Live agent was not given {k}")


In [2]:
import os

import ray
import supersuit as ss
from ray import tune
from ray.rllib.algorithms.ppo import PPOConfig
from ray.rllib.env.wrappers.pettingzoo_env import ParallelPettingZooEnv
from ray.rllib.models import ModelCatalog
from ray.rllib.models.torch.torch_modelv2 import TorchModelV2
from ray.tune.registry import register_env
from torch import nn

#from pettingzoo.butterfly import pistonball_v6


class CNNModelV2(TorchModelV2, nn.Module):
    def __init__(self, obs_space, act_space, num_outputs, *args, **kwargs):
        TorchModelV2.__init__(self, obs_space, act_space, num_outputs, *args, **kwargs)
        nn.Module.__init__(self)
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, [8, 8], stride=(4, 4)),
            nn.ReLU(),
            nn.Conv2d(32, 64, [4, 4], stride=(2, 2)),
            nn.ReLU(),
            nn.Conv2d(64, 64, [3, 3], stride=(1, 1)),
            nn.ReLU(),
            nn.Flatten(),
            (nn.Linear(3136, 512)),
            nn.ReLU(),
        )
        self.policy_fn = nn.Linear(512, num_outputs)
        self.value_fn = nn.Linear(512, 1)

    def forward(self, input_dict, state, seq_lens):
        model_out = self.model(input_dict["obs"].permute(0, 3, 1, 2))
        self._value_out = self.value_fn(model_out)
        return self.policy_fn(model_out), state

    def value_function(self):
        return self._value_out.flatten()


def env_creator(args):
    env = CustomEnvironment()
    #env = ss.color_reduction_v0(env, mode="B")
    #env = ss.dtype_v0(env, "float32")
    #env = ss.resize_v1(env, x_size=84, y_size=84)
    #env = ss.normalize_obs_v0(env, env_min=0, env_max=1)
    #env = ss.frame_stack_v1(env, 3)
    return env


if __name__ == "__main__":
    ray.init()

    env_name = "CustomEnvironment"

    register_env(env_name, lambda config: ParallelPettingZooEnv(env_creator(config)))
    ModelCatalog.register_custom_model("CNNModelV2", CNNModelV2)

    config = (
    PPOConfig()
    .environment(env=env_name, clip_actions=True)
    .env_runners(num_env_runners=4, rollout_fragment_length=128)
    .training(
        train_batch_size=512,
        lr=2e-5,
        gamma=0.99,
        lambda_=0.9,
        use_gae=True,
        clip_param=0.4,
        grad_clip=None,
        entropy_coeff=0.1,
        vf_loss_coeff=0.25,
    )
    .framework("torch")
    .debugging(log_level="ERROR")
    .resources(num_gpus=int(os.environ.get("RLLIB_NUM_GPUS", "0")))
    .update_from_dict({
        "num_sgd_iter": 10,
        "sgd_minibatch_size": 64,  # ✅ correct place
    })
)

    tune.run(
        "PPO",
        name="PPO",
        stop={"timesteps_total": 5000000 if not os.environ.get("CI") else 50000},
        checkpoint_freq=10,
        storage_path="~/ray_results/" + env_name,
        config=config.to_dict(),
    )

2025-10-05 15:31:47,516	INFO worker.py:1951 -- Started a local Ray instance.
2025-10-05 15:31:48,378	INFO tune.py:616 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949
/home/ethan/.local/lib/python3.10/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/ethan/.local/lib/python3.10/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/home/ethan/.local/lib/python3.10/site-packages/gymnasium/utils/passive_env_checker.py:134: UserWarning: WARN: The obs returned by the `reset()` method was expecting numpy array dtype to be float32, actual type: float64
  logger.warn(
/home/ethan/.local/lib/python3.10/site-packages/gymnas

2025-10-05 15:31:48,420	WARNING algorithm_config.py:5045 -- You are running PPO on the new API stack! This is the new default behavior for this algorithm. If you don't want to use the new API stack, set `config.api_stack(enable_rl_module_and_learner=False,enable_env_runner_and_connector_v2=False)`. For a detailed migration guide, see here: https://docs.ray.io/en/master/rllib/new-api-stack-migration-guide.html
(PPO pid=65895) 2025-10-05 15:31:50,693	WARNING algorithm_config.py:5045 -- You are running PPO on the new API stack! This is the new default behavior for this algorithm. If you don't want to use the new API stack, set `config.api_stack(enable_rl_module_and_learner=False,enable_env_runner_and_connector_v2=False)`. For a detailed migration guide, see here: https://docs.ray.io/en/master/rllib/new-api-stack-migration-guide.html
(PPO pid=65895) [2025-10-05 15:31:51,069 E 65895 65895] core_worker.cc:2246: Actor with class name: 'SingleAgentEnvRunner' and ID: '8cc2a896fa53fb25077f1fde01

(SingleAgentEnvRunner pid=65993) [37, 41, 75]
(SingleAgentEnvRunner pid=65993) self.agents: ['satellite1', 'satellite2', 'satellite3']
(SingleAgentEnvRunner pid=65993) Returning observations for: ['satellite1', 'satellite2', 'satellite3']
(SingleAgentEnvRunner pid=65993) Expected agents: ['satellite1', 'satellite2', 'satellite3']
(SingleAgentEnvRunner pid=65994) [21, 88, 15]
(SingleAgentEnvRunner pid=65995) [12, 17, 35]
(SingleAgentEnvRunner pid=65996) [68, 78, 74]


(SingleAgentEnvRunner pid=65993) 2025-10-05 15:31:53,964	WARNING rl_module.py:432 -- Didn't create a Catalog object for your RLModule! If you are not using the new API stack yet, make sure to switch it off in your config: `config.api_stack(enable_rl_module_and_learner=False, enable_env_runner_and_connector_v2=False)`. All algos use the new stack by default. Ignore this message, if your RLModule does not use a Catalog to build its sub-components.
(SingleAgentEnvRunner pid=65993) 2025-10-05 15:31:53,964	WARNING deprecation.py:50 -- DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=..)` instead. This will raise an error in the future!
(SingleAgentEnvRunner pid=65993) Exception raised in creation task: The actor died because of an error raised in its creation task, ray::SingleAgentEnvRunner.__init__() (pid=65993, ip=192.168.2.33, actor_id=8cc2a896fa53fb25

Trial name
PPO_CustomEnvironment_ef7c7_00000


2025-10-05 15:31:54,149	INFO tune.py:1009 -- Wrote the latest version of all result files and experiment state to '/home/ethan/ray_results/CustomEnvironment/PPO' in 0.0034s.
(PPO pid=65895) 2025-10-05 15:31:54,119	ERROR actor_manager.py:873 -- Ray error (The actor died because of an error raised in its creation task, ray::SingleAgentEnvRunner.__init__() (pid=65993, ip=192.168.2.33, actor_id=8cc2a896fa53fb25077f1fde01000000, repr=<ray.rllib.env.single_agent_env_runner.SingleAgentEnvRunner object at 0x72062beb00d0>)
(PPO pid=65895) 
(PPO pid=65895) 
(PPO pid=65895) ValueError: No default encoder config for obs space=Dict('satellite1': Dict('action_mask': MultiBinary(2), 'central_observation_matrix': Box(0.0, 1.0, (30, 5), float32), 'current_timestep': Box(0.0, 32.0, (1,), float32), 'remaining_satellite_data': Box(-1.0, 1.0, (3,), float32)), 'satellite2': Dict('action_mask': MultiBinary(2), 'central_observation_matrix': Box(0.0, 1.0, (30, 5), float32), 'current_timestep': Box(0.0, 32.0, (

TuneError: ('Trials did not complete', [PPO_CustomEnvironment_ef7c7_00000])

(raylet) [2025-10-05 17:34:47,554 E 64549 64549] (raylet) node_manager.cc:2929: 17 Workers (tasks / actors) killed due to memory pressure (OOM), 0 Workers crashed due to other reasons at node (ID: dc787a8feed54e0e9741a98875a19bd03ba58cba9d95b1a5cdf426dc, IP: 192.168.2.33) over the last time period. To see more information about the Workers killed on this node, use `ray logs raylet.out -ip 192.168.2.33`
(raylet) 
(raylet) Refer to the documentation on how to address the out of memory issue: https://docs.ray.io/en/latest/ray-core/scheduling/ray-oom-prevention.html. Consider provisioning more memory on this node or reducing task parallelism by requesting more CPUs per task. To adjust the kill threshold, set the environment variable `RAY_memory_usage_threshold` when starting Ray. To disable worker killing, set the environment variable `RAY_memory_monitor_refresh_ms` to zero.
